# Monday Morning Borough Briefing
Scheduled job for the Monday 10:00 borough-agency call: one borough + one complaint type → a seven-column on-screen list with a derived `is_open` flag.

Order of work: (1) pick parameters, (2) prototype the open/closed rule on 4 invented complaints, (3) run the briefing at scale from the CSV header.

In [8]:
import re
import time
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, types as T

# ---- parameters (the only thing the analyst changes) ----
BOROUGH        = "BROOKLYN"
COMPLAINT_TYPE = "Street Light Condition"
ROWS_ON_SCREEN = 20

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_PATH = str(ROOT / "data" / "311-service-requests-from-2010-to-present.csv")

spark = (SparkSession.builder.master("local[*]").config("spark.driver.memory", "8g")
         .appName("borough-briefing").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| data:", DATA_PATH)

Spark 4.2.0 | data: /Users/hs086u/Documents/learning/terraform-projects/nyc-311-data-pipeline/se-spark/data/311-service-requests-from-2010-to-present.csv


## 1. Custom dataframes for open/closed rule

In [9]:
def add_is_open(df):
    # closed = status Closed AND a closed_date; null-safe, so unknowns stay open (never silently "closed")
    is_closed = F.coalesce((F.col("status") == F.lit("Closed")) & F.col("closed_date").isNotNull(), F.lit(False))
    return df.withColumn("is_open", ~is_closed)

proto = spark.createDataFrame([
    ("A: open, no close date",         "Open",   None),
    ("B: closed normally",             "Closed", "2026-01-05 09:00:00"),
    ("C: Closed but date missing",     "Closed", None),
    ("D: Pending but has closed_date", "Pending","2026-01-06 09:00:00"),
], "case string, status string, closed_date string")

add_is_open(proto).show(truncate=False)

+------------------------------+-------+-------------------+-------+
|case                          |status |closed_date        |is_open|
+------------------------------+-------+-------------------+-------+
|A: open, no close date        |Open   |NULL               |true   |
|B: closed normally            |Closed |2026-01-05 09:00:00|false  |
|C: Closed but date missing    |Closed |NULL               |true   |
|D: Pending but has closed_date|Pending|2026-01-06 09:00:00|true   |
+------------------------------+-------+-------------------+-------+



## 2. Briefing logic to run on passed dataframes

In [10]:
def briefing(df, borough, complaint_type):
    return (add_is_open(df)
            .filter((F.col("borough") == borough) & (F.col("complaint_type") == complaint_type))
            .select("complaint_type", "descriptor", "agency", "created_date", "closed_date", "status", "is_open")
            .orderBy(F.col("created_date").desc()))


## 3a. Run the briefing: header read, no inference
Column names come from the CSV header; Spark skips the type-guessing scan. This is the run that goes on screen.

In [11]:
# Column names come from the CSV header row (snake_cased so the rest of the notebook stays readable);
# nothing is hardcoded, so a reordered or extended file still lines up.
import csv
with open(DATA_PATH, newline="") as f:
    HEADER = next(csv.reader(f))                            # plain Python: no JVM needed
SCHEMA_COLS = [re.sub(r"\W+", "_", h.strip()).strip("_").lower() for h in HEADER]

def run_job(reader, borough, complaint_type):
    t0 = time.perf_counter()
    result = briefing(reader(), borough, complaint_type)   # reader() is inside the timer: inference happens at read time
    total_matches = result.count()
    open_n = result.filter("is_open").count()
    screen = result.limit(ROWS_ON_SCREEN)
    screen.collect()
    return screen, total_matches, open_n, time.perf_counter() - t0

read_declared = lambda: spark.read.csv(DATA_PATH, header=True, inferSchema=False).toDF(*SCHEMA_COLS)

screen, total_matches, open_n, t_declared = run_job(read_declared, BOROUGH, COMPLAINT_TYPE)

print(f"{BOROUGH} | {COMPLAINT_TYPE}: {total_matches:,} complaints, {open_n:,} still open "
      f"-- showing latest {ROWS_ON_SCREEN}")
screen.show(ROWS_ON_SCREEN, truncate=30)
print(f"Runtime, header read: {t_declared:.2f}s")


BROOKLYN | Street Light Condition: 88,510 complaints, 9,284 still open -- showing latest 20


+----------------------+------------------------------+------+-----------------------+-----------+------+-------+
|        complaint_type|                    descriptor|agency|           created_date|closed_date|status|is_open|
+----------------------+------------------------------+------+-----------------------+-----------+------+-------+
|Street Light Condition|              Street Light Out|   DOT|2019-11-30T23:23:00.000|       NULL|  Open|   true|
|Street Light Condition|              Street Light Out|   DOT|2019-11-30T21:26:00.000|       NULL|  Open|   true|
|Street Light Condition|              Street Light Out|   DOT|2019-11-30T18:50:00.000|       NULL|  Open|   true|
|Street Light Condition|              Street Light Out|   DOT|2019-11-30T18:44:00.000|       NULL|  Open|   true|
|Street Light Condition|              Street Light Out|   DOT|2019-11-30T18:16:00.000|       NULL|  Open|   true|
|Street Light Condition| Lamppost Base Door/Cover Open|   DOT|2019-11-30T17:38:00.000|  

## 3b. Run the briefing: `inferSchema=True`
Same job, but Spark scans the file once to guess types, then reads it again to load it.

In [12]:
read_infer = lambda: spark.read.csv(DATA_PATH, header=True, inferSchema=True).toDF(*SCHEMA_COLS)
_, total_matches_i, open_n_i, t_infer = run_job(read_infer, BOROUGH, COMPLAINT_TYPE)

assert (total_matches_i, open_n_i) == (total_matches, open_n)   # same answer either way
print(f"Runtime, inferSchema=True: {t_infer:.2f}s")
print(f"Header read {t_declared:.2f}s vs inferSchema {t_infer:.2f}s -> inference is {t_infer / t_declared:.1f}x slower")

Runtime, inferSchema=True: 32.32s
Header read 22.16s vs inferSchema 32.32s -> inference is 1.5x slower
